In [1]:

import os
import re
import pandas as pd
import platform
from datetime import datetime, timedelta
from scipy.io import wavfile
import numpy as np
from tqdm import tqdm
import sys
print(sys.executable)
import ast
import platform
from pathlib import Path

if platform.system() == "Windows":
    base_raw = Path(r"\\sanesstorage.cns.nyu.edu\archive\ginosar\Raw_data")
    base_processed = Path(r"\\sanesstorage.cns.nyu.edu\archive\ginosar\Processed_data")
else:
    base_raw = Path("/mnt/home/neurostatslab/ceph/saneslab_data/big_setup/")
    base_processed = Path("/mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Processed_data")

print("I'm at: ",base_raw)

/mnt/home/gginosar/repos/gerbil_vocalization_analysis/.venv/bin/python
I'm at:  /mnt/home/neurostatslab/ceph/saneslab_data/big_setup


In [ ]:
import pandas as pd
from pathlib import Path

# === INPUT / OUTPUT FOLDERS ===
input_folder = Path(fr"\\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\all_vox\TRAINING_data_NEW")
output_folder = Path(fr"\\sanesstorage.cns.nyu.edu\archive\ginosar\Pre_processing\das\models\all_vox\TRAINING_data_NEW_grouped")

output_folder.mkdir(parents=True, exist_ok=True)

# === DEFINE MAPPING ===
mapping = {
    "ufm": "high-freq",
    "tilda": "high-freq",
    "hat": "high-freq",
    "dfm": "high-freq",

    "dense-stack": "dense-stack",
    "low": "dense-stack",
    "growl": "dense-stack",
    "dfm-stack": "dense-stack",

    "mnt": "mnt",
    "half-mnt": "mnt",
}

# === PROCESS ALL CSV FILES ===
for csv_file in input_folder.glob("*.csv"):
    print(f"Processing: {csv_file.name}")

    df = pd.read_csv(csv_file)

    # --- Apply mapping ---
    df["name"] = df["name"].apply(lambda x: mapping.get(x, x))

    # --- Save to new folder ---
    out_path = output_folder / csv_file.name
    df.to_csv(out_path, index=False)

print("Done!")

In [22]:
exp = 518

def get_experiment_month(exp):
    if 97 <= exp <= 116:
        return "2024_12"
    elif 235 <= exp <= 237:
        return "2025_03"
    elif 272 <= exp < 278:
        return "2025_07"
    elif 332 <= exp < 344:
        return "2025_10"
    elif exp >= 491:
        return "2026_02"

    else:
        raise ValueError(f"Unknown experiment range for {exp}")
month_folder = get_experiment_month(exp)


# Build paths
folder_path_raw_wavs = base_raw / f"experiment_{exp}" / "concatenated_data_cam_mic_sync" #/ "wavs"
folder_path_sync = base_raw / f"experiment_{exp}" / "concatenated_data_cam_mic_sync"
folder_path_averaged_wavs = base_processed / "Audio" / month_folder / str(exp) / "Averaged_wavs_w_annotations"
folder_path_Audio = base_processed / "Audio" / month_folder / str(exp)

# Optional: print to verify
print(folder_path_raw_wavs)
print(folder_path_averaged_wavs)
print(folder_path_Audio)


# --- Parameters ----------------------------------------------------
#arena_wav_folder = folder_path_averaged_wavs  
sample_rate_expected = 125000
#output_folder = folder_path_Audio  #  output path for CSVs

time_window_overlapping_calls_sec = 0.015
inter_call_interval_within_bout_threshold = 10.0  # seconds
min_calls_per_bout = 5 


# ##############################################################################
# load sync file
# ##############################################################################

def load_sync_file(exp, folder_path_sync):
    sync_path = Path(folder_path_sync) / "sync.csv"
    
    if not sync_path.exists():
        raise FileNotFoundError(f"Sync file not found: {sync_path}")

    sync_df = pd.read_csv(sync_path)
    print(list(sync_df.columns))
    
    # parse the time lists
    sync_df['timestamp'] = sync_df['timestamp'].apply(ast.literal_eval)
   
    # chaneg to date object
    sync_df['start_time'] = pd.to_datetime(sync_df['timestamp'].str[0])
    start_time_experiment = sync_df.iloc[0]['start_time']
    print(f"Experiment {exp} started at: {start_time_experiment}")

    return start_time_experiment, sync_df


# Load sync file 
start_time_experiment,sync_df = load_sync_file(exp,folder_path_sync)
#sync_df.head()


/mnt/home/neurostatslab/ceph/saneslab_data/big_setup/experiment_518/concatenated_data_cam_mic_sync
/mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Processed_data/Audio/2026_02/518/Averaged_wavs_w_annotations
/mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Processed_data/Audio/2026_02/518
['video', 'audio', 'timestamp']
Experiment 518 started at: 2026-02-27 14:47:13


In [ ]:
# 0)------ Avergae audio data ----- > average two microphones for each arena
# ##############################################################################

def get_channel_mapping(exp):

    if exp < 272:
        return {
            "10": [0, 1],
            "20": [2, 3],
            "30": [4, 5],
        }
    else: # => 272
        return {
            "10": [2, 3],
            "20": [4, 5],
            "30": [0, 1],
        }


def average_microphone_pairs(exp, input_folder, output_folder):

    input_folder = Path(input_folder)
    output_folder = Path(output_folder)

    # Create the output folder if it does not already exist
    output_folder.mkdir(parents=True, exist_ok=True)

    # get the correct channel mapping for this experiment
    channel_mapping = get_channel_mapping(exp)

    print(f"\n--- Processing experiment {exp} ---")
    print(f"Reading raw WAV files from: {input_folder}")
    print(f"Saving averaged WAV files to: {output_folder}")
    print("Using channel mapping:")

    for virtual_ch, real_pair in channel_mapping.items():
        print(f"  virtual channel {virtual_ch} <- real channels {real_pair}")

    # --------------------------------------------------
    # STEP 1: Find all file numbers
    #
    # We use channel_00 files as a reference to find which
    # file chunks exist: 001, 002, 003, ...
    # --------------------------------------------------

    file_nums = []

    # Compile the regex once
    pattern = re.compile(r"channel_00_file_(\d+)\.wav")

    for file_path in input_folder.iterdir():
        match = pattern.match(file_path.name)

        if match:
            # Extract file number, e.g. "001" -> 1
            file_num = int(match.group(1))
            file_nums.append(file_num)

    # Sort file numbers so they are processed in order
    file_nums = sorted(file_nums)

    print(f"Found {len(file_nums)} file chunks")

    # Counters for summary at the end
    n_saved = 0
    n_missing = 0
    n_shape_mismatch = 0
    n_rate_mismatch = 0

    # --------------------------------------------------
    # STEP 2: Loop over each file number
    # --------------------------------------------------

    for file_num in file_nums:

        # Format with leading zeros: 1 -> "001"
        file_num_str = f"{file_num:03d}"

        # --------------------------------------------------
        # STEP 3: For this file number, create each virtual channel
        # --------------------------------------------------

        for virtual_ch, real_pair in channel_mapping.items():

            # real_pair is something like [2, 3]
            ch1, ch2 = real_pair
            #print(f"\nFile {file_num_str}: averaging channels {ch1} and {ch2} -> virtual channel {virtual_ch}")

            # Build the full paths to the two real microphone files
            path1 = folder_path_raw_wavs / f"channel_{ch1:02d}_file_{file_num_str}.wav"
            path2 = folder_path_raw_wavs / f"channel_{ch2:02d}_file_{file_num_str}.wav"

            # --------------------------------------------------
            # STEP 4: Check that both files exist
            # --------------------------------------------------

            if not path1.exists() or not path2.exists():
                print("  Skipping because one or both files are missing")
                print(f"  Expected file: {path1.name}")
                print(f"  Expected file: {path2.name}")
                n_missing += 1
                continue

            # --------------------------------------------------
            # STEP 5: Read both WAV files
            # --------------------------------------------------

            rate1, data1 = wavfile.read(path1) #data = waveform samples as a numpy array
            rate2, data2 = wavfile.read(path2)

            # --------------------------------------------------
            # STEP 6: Check that the two files are compatible
            # --------------------------------------------------

            if rate1 != rate2:
                print(f"  Skipping because sample rates differ: {rate1} vs {rate2}")
                n_rate_mismatch += 1
                continue

            if data1.shape != data2.shape:
                print(f"  Skipping because shapes differ: {data1.shape} vs {data2.shape}")
                n_shape_mismatch += 1
                continue

            # --------------------------------------------------
            # STEP 7: Average the two signals
            # We convert to float32
            # --------------------------------------------------

            data1 = data1.astype(np.float32, copy=False)
            data2 = data2.astype(np.float32, copy=False)

            avg_data = (data1 + data2) / 2.0

            # --------------------------------------------------
            # STEP 8: Save the averaged output
            # --------------------------------------------------

            out_path = output_folder / f"channel_{virtual_ch}_file_{file_num_str}.wav"

            wavfile.write(out_path, rate1, avg_data)

            #print(f"  Saved: {out_path.name}")
            n_saved += 1

    # --------------------------------------------------
    # STEP 9: Print summary
    # --------------------------------------------------

    print("\n--- Done ---")
    print(f"Saved files: {n_saved}")
    print(f"Skipped because of missing files: {n_missing}")
    print(f"Skipped because of shape mismatch: {n_shape_mismatch}")
    print(f"Skipped because of sample rate mismatch: {n_rate_mismatch}")

average_microphone_pairs(exp, folder_path_raw_wavs, folder_path_averaged_wavs)


--- Processing experiment 518 ---
Reading raw WAV files from: /mnt/home/neurostatslab/ceph/saneslab_data/big_setup/experiment_518/concatenated_data_cam_mic_sync
Saving averaged WAV files to: /mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Processed_data/Audio/2026_02/518/Averaged_wavs_w_annotations
Using channel mapping:
  virtual channel 10 <- real channels [2, 3]
  virtual channel 20 <- real channels [4, 5]
  virtual channel 30 <- real channels [0, 1]
Found 26 file chunks

File 000: averaging channels 2 and 3 -> virtual channel 10
  Saved: channel_10_file_000.wav

File 000: averaging channels 4 and 5 -> virtual channel 20
  Saved: channel_20_file_000.wav

File 000: averaging channels 0 and 1 -> virtual channel 30
  Saved: channel_30_file_000.wav

File 001: averaging channels 2 and 3 -> virtual channel 10
  Saved: channel_10_file_001.wav

File 001: averaging channels 4 and 5 -> virtual channel 20
  Saved: channel_20_file_001.wav

File 001: averaging channels 0 and 1 -> virtual c

KeyboardInterrupt: 

In [1]:
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy.signal import spectrogram

# ============================================================
# SETTINGS
# ============================================================

csv_folder = Path(fr"/mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Processed_data/Audio/2026_02/530/Averaged_wavs_w_annotations")
audio_folder = Path(fr"/mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Processed_data/Audio/2026_02/530/Averaged_wavs_w_annotations")   # can contain .wav and/or .npz
output_folder = Path(fr"/mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Processed_data/Audio/2026_02/530/viz")
output_folder.mkdir(parents=True, exist_ok=True)

# CSV column names
label_col = "name"
start_col = "start_seconds"
stop_col = "stop_seconds"

# Spectrogram settings
buffer_sec = 0
nperseg = 512
noverlap = 256
spec_min_val = -8.0
spec_max_val = -5.0

# Plot settings
max_examples_per_label = None   # e.g. 200, or None for all
examples_per_page = 100
cols = 10
figsize_per_panel = 1.5


# ============================================================
# HELPERS
# ============================================================

def is_real_event_row(df: pd.DataFrame) -> pd.Series:
    """
    Keep only rows that contain a real event.
    Removes DAS template/empty rows.
    """
    start = pd.to_numeric(df[start_col], errors="coerce")
    stop = pd.to_numeric(df[stop_col], errors="coerce")

    return (
        start.notna()
        & stop.notna()
        & (stop > start)
        & (start > 0)
    )


def clean_label(label: str) -> str:
    return str(label).strip()


def find_matching_audio(csv_path: Path) -> Path | None:
    """
    Match:
      exp_272_channel_30_file_063_annotations.csv
    to either:
      exp_272_channel_30_file_063.wav
      exp_272_channel_30_file_063.npz
    """
    stem = csv_path.stem

    if stem.endswith("_annotations"):
        stem = stem[:-12]   # remove "_annotations"

    wav_path = audio_folder / f"{stem}.wav"
    if wav_path.exists():
        return wav_path

    npz_path = audio_folder / f"{stem}.npz"
    if npz_path.exists():
        return npz_path

    return None


def load_audio_file(audio_path: Path):
    """
    Load either .wav or .npz audio.

    Returns
    -------
    fs : int or float
    audio : np.ndarray
    """
    suffix = audio_path.suffix.lower()

    if suffix == ".wav":
        fs, audio = wavfile.read(audio_path)
        return fs, audio

    elif suffix == ".npz":
        data = np.load(audio_path, allow_pickle=True)

        print(f"[info] npz keys in {audio_path.name}: {data.files}")

        possible_audio_keys = ["audio", "wav", "waveform", "x", "samples"]
        possible_fs_keys = ["fs", "sr", "samplerate", "sample_rate"]

        audio = None
        fs = None

        for k in possible_audio_keys:
            if k in data.files:
                audio = data[k]
                break

        for k in possible_fs_keys:
            if k in data.files:
                fs = data[k]
                break

        if audio is None:
            raise ValueError(
                f"Could not find audio array in {audio_path.name}. "
                f"Available keys: {data.files}"
            )

        if fs is None:
            raise ValueError(
                f"Could not find sample rate in {audio_path.name}. "
                f"Available keys: {data.files}"
            )

        if isinstance(fs, np.ndarray):
            fs = fs.item()

        return fs, audio

    else:
        raise ValueError(f"Unsupported audio type: {audio_path}")


def make_spec(signal: np.ndarray, fs: int):
    """
    Return an AVA-style normalized log spectrogram for plotting.
    """
    if signal.ndim > 1:
        signal = signal[:, 0]

    if len(signal) < nperseg:
        signal = np.pad(signal, (0, nperseg - len(signal)))

    f, t, Sxx = spectrogram(
        signal,
        fs=fs,
        nperseg=nperseg,
        noverlap=noverlap,
        scaling="spectrum",
        mode="magnitude",
    )

    Sxx = np.log(Sxx + 1e-12)
    Sxx = (Sxx - spec_min_val) / (spec_max_val - spec_min_val)
    Sxx = np.clip(Sxx, 0.0, 1.0)
    return f, t, Sxx


# ============================================================
# STEP 1: COLLECT ALL CALLS BY LABEL
# ============================================================

calls_by_label = {}

csv_files = sorted(csv_folder.glob("*.csv"))
print(f"Found {len(csv_files)} CSV files")

for csv_file in csv_files:
    try:
        df = pd.read_csv(csv_file)
    except Exception as e:
        print(f"[skip] could not read {csv_file.name}: {e}")
        continue

    needed = {label_col, start_col, stop_col}
    if not needed.issubset(df.columns):
        print(f"[skip] missing columns in {csv_file.name}")
        continue

    df = df[is_real_event_row(df)].copy()
    if len(df) == 0:
        continue

    audio_path = find_matching_audio(csv_file)
    if audio_path is None:
        print(f"[skip] no matching audio for {csv_file.name}")
        continue

    df[start_col] = pd.to_numeric(df[start_col], errors="coerce")
    df[stop_col] = pd.to_numeric(df[stop_col], errors="coerce")

    for _, row in df.iterrows():
        label = clean_label(row[label_col])

        event = {
            "csv_file": csv_file,
            "audio_file": audio_path,
            "label": label,
            "start": float(row[start_col]),
            "stop": float(row[stop_col]),
            "duration": float(row[stop_col] - row[start_col]),
        }

        calls_by_label.setdefault(label, []).append(event)

print("\nCollected calls by label:")
for label, events in sorted(calls_by_label.items()):
    print(f"  {label}: {len(events)} calls")


# ============================================================
# STEP 2: PLOT EACH LABEL
# ============================================================

for label, events in sorted(calls_by_label.items()):
    if max_examples_per_label is not None:
        events = events[:max_examples_per_label]

    if len(events) == 0:
        continue

    print(f"\nPlotting label '{label}' with {len(events)} calls")

    label_out = output_folder / label
    label_out.mkdir(parents=True, exist_ok=True)

    # Group by audio file so we don't reread the same file too many times
    events_by_audio = {}
    for ev in events:
        events_by_audio.setdefault(ev["audio_file"], []).append(ev)

    all_specs = []

    for audio_path, audio_events in events_by_audio.items():
        try:
            fs, audio = load_audio_file(audio_path)
        except Exception as e:
            print(f"[skip audio] {audio_path.name}: {e}")
            continue

        if audio.ndim > 1:
            audio = audio[:, 0]

        n_samples = len(audio)

        for ev in audio_events:
            start = max(0, ev["start"] - buffer_sec)
            stop = ev["stop"] + buffer_sec

            s0 = int(round(start * fs))
            s1 = int(round(stop * fs))
            s1 = min(s1, n_samples)

            if s1 <= s0:
                continue

            segment = audio[s0:s1]

            try:
                f, t, Sxx = make_spec(segment, fs)
            except Exception as e:
                print(f"[skip spec] {audio_path.name}: {e}")
                continue

            all_specs.append({
                "spec": Sxx,
                "event": ev,
            })

    if len(all_specs) == 0:
        print(f"[skip label] no spectrograms made for {label}")
        continue

    num_pages = math.ceil(len(all_specs) / examples_per_page)

    for page_idx in range(num_pages):
        start_i = page_idx * examples_per_page
        end_i = min((page_idx + 1) * examples_per_page, len(all_specs))
        page_specs = all_specs[start_i:end_i]

        n = len(page_specs)
        ncols = cols
        nrows = math.ceil(n / ncols)

        fig, axes = plt.subplots(
            nrows, ncols,
            figsize=(ncols * figsize_per_panel, nrows * figsize_per_panel)
        )

        if nrows == 1 and ncols == 1:
            axes = np.array([[axes]])
        elif nrows == 1:
            axes = np.array([axes])
        elif ncols == 1:
            axes = axes[:, np.newaxis]

        axes = axes.flatten()

        for ax, item in zip(axes, page_specs):
            Sxx = item["spec"]
            ev = item["event"]

            ax.imshow(
                Sxx,
                aspect="auto",
                origin="lower",
                interpolation="nearest",
                vmin=0.0,
                vmax=1.0,
            )
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_title(ev["audio_file"].name, fontsize=4)

        for ax in axes[len(page_specs):]:
            ax.axis("off")

        fig.suptitle(
            f"{label} — page {page_idx + 1}/{num_pages} — n={len(page_specs)}",
            fontsize=14
        )
        fig.tight_layout()

        out_path = label_out / f"{label}_page_{page_idx + 1:03d}.png"
        fig.savefig(out_path, dpi=200, bbox_inches="tight")
        plt.close(fig)

    print(f"Saved {num_pages} page(s) for label '{label}' to {label_out}")

print("\nDone.")

Found 57 CSV files

Collected calls by label:
  alarm: 52 calls
  dense-stack: 4728 calls
  dfm-stack: 497 calls
  high-freq: 3651 calls
  mnt: 7 calls
  newborn: 114 calls
  noise: 19990 calls
  warble: 2339 calls

Plotting label 'alarm' with 52 calls
Saved 1 page(s) for label 'alarm' to /mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Processed_data/Audio/2026_02/530/viz/alarm

Plotting label 'dense-stack' with 4728 calls
Saved 48 page(s) for label 'dense-stack' to /mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Processed_data/Audio/2026_02/530/viz/dense-stack

Plotting label 'dfm-stack' with 497 calls
Saved 5 page(s) for label 'dfm-stack' to /mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Processed_data/Audio/2026_02/530/viz/dfm-stack

Plotting label 'high-freq' with 3651 calls
Saved 37 page(s) for label 'high-freq' to /mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Processed_data/Audio/2026_02/530/viz/high-freq

Plotting label 'mnt' with 7 calls
Saved 1 page(s) f